# Calibration Count Bias: one GPU confirmation run

**Run all cells once. No prior result upload, manual dataset preparation, or separate sweeps are needed.**

Suggested host: one NVIDIA A100 with 80 GB GPU memory, at least 32 GB system RAM and 100 GB free persistent disk, Linux, Python 3.10 to 3.12, and a CUDA 12.6 compatible driver. These are provisioning targets, not measured minimum requirements. A clean isolated Python environment is created automatically.

This notebook downloads the official test pools of CIFAR10, Food101, and FGVC Aircraft and evaluates three pinned CLIP/SigLIP encoders. It runs the frozen coverage, complete prediction set, baseline, and screening protocol. It saves every scheduled condition and returns one result archive. The encoder stage needs the GPU; statistical and ridge computations also use the CPU.

The source code was syntax checked and its checkpoint, batching, dataset, and analysis contracts were tested locally with synthetic mock inputs. **The real GPU extraction and these three new dataset experiments have not been executed by the assistant.** The manuscript does not claim their results. Successful execution supplies new empirical evidence, not an acceptance probability.

Keep the working directory on persistent storage. Restarting this notebook with unchanged code, environment, and directory resumes verified image feature checkpoints. An interrupted statistical phase restarts that phase using the saved features. Do not alter parameters or omit unfavorable datasets after seeing results.

The final archive to return is `Calibration_Count_NEW_CONDITIONS_RESULTS.zip`. On failure, return `Calibration_Count_DIAGNOSTICS.zip` and keep the working directory.

**v2 recovery note:** before the frozen runner starts, the notebook scans only derived selected-image PNG snapshots and removes any unreadable partial files so the unchanged runner can regenerate them deterministically from the downloaded dataset. This does not change the scientific protocol.
> **v3 notebook wrapper:** the scientific runner and protocol are unchanged. This notebook only retries when the persistent network filesystem exposes an unreadable derived `data/selected/*.png` snapshot. It deletes only that unreadable derived PNG and reruns the identical frozen runner. Any other error stops immediately.


## 1. Verify the embedded code and prepare the isolated environment

In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, shutil, subprocess, sys, time, zipfile

# Change only this path when a persistent disk is mounted somewhere else.
WORKDIR = Path(os.environ.get("CALIBRATION_ROOT", str(Path.cwd() / "calibration_count_gpu"))).resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)
SOURCE = WORKDIR / "runner"
SOURCE.mkdir(exist_ok=True)
PAYLOAD_SHA256 = "c85a9951cd169716e54e5564df1d70f008cb336d8c55f5de86472906e642bb58"
PAYLOAD = "..."
raw = base64.b64decode(PAYLOAD)
if hashlib.sha256(raw).hexdigest() != PAYLOAD_SHA256:
    raise RuntimeError("Embedded source archive failed its integrity check.")
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for name in z.namelist():
        if Path(name).name != name:
            raise RuntimeError("Unsafe archive path.")
        path = SOURCE / name
        content = z.read(name)
        if path.exists() and path.read_bytes() != content:
            raise RuntimeError(f"Existing source differs: {path}. Preserve the old run and use a new WORKDIR.")
        path.write_bytes(content)
print("Verified embedded scientific code:", PAYLOAD_SHA256)
print("Persistent working directory:", WORKDIR)

# An isolated environment prevents notebook-image Torch/NumPy conflicts.
candidates = [sys.executable] + [shutil.which(n) for n in ("python3.12", "python3.11", "python3.10")]
BASE = None
for exe in dict.fromkeys(x for x in candidates if x):
    try:
        version = json.loads(subprocess.check_output([exe, "-c", "import sys,json;print(json.dumps(list(sys.version_info[:2])))"], text=True))
        if version[0] == 3 and 10 <= version[1] <= 12:
            BASE = exe
            break
    except Exception:
        continue
if BASE is None:
    raise RuntimeError("Use a Linux GPU notebook with Python 3.10, 3.11, or 3.12. The scientific scripts are preserved; no data were run.")
VENV = WORKDIR / "venv"
PYTHON = VENV / "bin" / "python"
if not PYTHON.exists():
    subprocess.run([BASE, "-m", "venv", str(VENV)], check=True)

ENV = os.environ.copy()
ENV.update(OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1",
           NUMEXPR_NUM_THREADS="1", TOKENIZERS_PARALLELISM="false",
           HF_HUB_DISABLE_XET="1", PYTHONUNBUFFERED="1")

def live(command, log_path):
    """Keep complete logs and stream progress; propagate every failure."""
    with Path(log_path).open("a", encoding="utf-8") as log:
        log.write("\nCOMMAND: " + repr([str(v) for v in command]) + "\n")
        process = subprocess.Popen([str(v) for v in command], stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1, env=ENV)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line); log.flush()
            code = process.wait()
        except BaseException:
            process.terminate()
            try: process.wait(timeout=15)
            except subprocess.TimeoutExpired: process.kill()
            raise
        if code != 0:
            raise RuntimeError(f"Command exited with {code}. Full log: {log_path}")

setup_identity = hashlib.sha256((PAYLOAD_SHA256 + "torch2.7.1+cu126|torchvision0.22.1").encode()).hexdigest()
setup_receipt = WORKDIR / "SETUP_COMPLETE.json"
if not setup_receipt.exists():
    live([PYTHON, "-m", "pip", "install", "--upgrade", "pip"], WORKDIR / "setup.log")
    live([PYTHON, "-m", "pip", "install", "torch==2.7.1", "torchvision==0.22.1",
          "--index-url", "https://download.pytorch.org/whl/cu126"], WORKDIR / "setup.log")
    live([PYTHON, "-m", "pip", "install", "-r", SOURCE / "requirements.txt"], WORKDIR / "setup.log")
    live([PYTHON, "-m", "pip", "check"], WORKDIR / "setup.log")
    setup_receipt.write_text(json.dumps({"identity": setup_identity, "python": str(PYTHON)}, indent=2))
else:
    if json.loads(setup_receipt.read_text())["identity"] != setup_identity:
        raise RuntimeError("The existing environment belongs to another frozen source. Use a separate WORKDIR.")
    live([PYTHON, "-m", "pip", "check"], WORKDIR / "setup.log")
print("Isolated environment ready. No kernel restart is needed.")


## 2. Check the GPU, package interfaces, and resumable runner

In [ ]:
check = r"""
import importlib.metadata,json,sys
from pathlib import Path
sys.path.insert(0,sys.argv[1])
from pinned_encoder_helpers import configure_torch, import_models
import torchvision
assert importlib.metadata.version('torch').split('+')[0]=='2.7.1'
assert importlib.metadata.version('torchvision').split('+')[0]=='0.22.1'
torch,env=configure_torch()
AP,C,S=import_models()
a=torch.randn(128,128,device='cuda:0')
assert torch.isfinite(a@a).all().item()
assert torch.isfinite(torch.exp(a.double().clamp(-5,5))).all().item()
print(json.dumps(env,indent=2))
Path(sys.argv[2]).write_text(json.dumps(env,indent=2))
"""
live([PYTHON, "-c", check, SOURCE, WORKDIR / "GPU_PREFLIGHT.json"], WORKDIR / "preflight.log")
live([PYTHON, SOURCE / "test_runner_local.py", WORKDIR / "LOCAL_CONTRACT_TESTS.json"], WORKDIR / "preflight.log")
print("GPU matrix test, model API imports, and local runner contracts passed.")


## 3. Run the complete frozen experiment

In [ ]:
# Repair only derived selected-image snapshots that are unreadable.
# This does not change the frozen protocol, dataset identities, model settings, or runner source.
RUN = WORKDIR / "confirmation"
repair_script = r'''
from pathlib import Path
import sys
from PIL import Image
root = Path(sys.argv[1]) / 'data' / 'selected'
bad = []
if root.exists():
    for p in sorted(root.rglob('*.png')):
        try:
            with Image.open(p) as im:
                im.load()
                if im.mode not in ('RGB','RGBA','L','P','CMYK','YCbCr','I','F'):
                    raise ValueError(f'unexpected image mode {im.mode}')
        except Exception as e:
            bad.append((p, repr(e)))
for p, err in bad:
    print('Removing unreadable derived snapshot:', p, err)
    p.unlink(missing_ok=True)
print(f'Snapshot repair complete: removed {len(bad)} unreadable derived PNG(s).')
'''
live([PYTHON, '-c', repair_script, RUN], WORKDIR / 'snapshot_repair.log')
print('Safe to resume the unchanged frozen runner.')


In [ ]:
RUN = WORKDIR / "confirmation"

def remove_unreadable_selected_images(run_root):
    # Only repair derived PNG snapshots. IDs, labels, source datasets, protocol,
    # model settings, and the frozen runner remain unchanged.
    from PIL import Image
    root = Path(run_root) / "data" / "selected"
    removed = []
    if not root.exists():
        return removed
    for p in sorted(root.rglob("*.png")):
        try:
            with Image.open(p) as im:
                im.load()
        except Exception as exc:
            removed.append((str(p), repr(exc)))
            p.unlink(missing_ok=True)
    return removed

# Network file systems can occasionally expose a partially committed derived PNG
# immediately after PIL closes it. The frozen runner already verifies that every
# successful snapshot decodes and is pixel-identical to its source image. Therefore
# it is safe to delete only unreadable derived snapshots and rerun the identical
# frozen runner until all snapshots commit correctly. Scientific errors are NOT retried.
MAX_SNAPSHOT_RETRIES = 12
attempt = 0
while True:
    attempt += 1
    removed = remove_unreadable_selected_images(RUN)
    if removed:
        print(f"Before attempt {attempt}: removed {len(removed)} unreadable derived snapshot(s)")
        for path, err in removed[:20]:
            print("  ", path, err)

    try:
        live([PYTHON, SOURCE / "confirm_once.py", "--root", RUN], WORKDIR / "RUN.log")
        break
    except BaseException:
        status_path = RUN / "STATUS.json"
        status = {}
        if status_path.exists():
            try:
                status = json.loads(status_path.read_text())
            except Exception:
                pass
        tb = str(status.get("traceback", ""))
        err = str(status.get("error", ""))
        snapshot_failure = (
            "UnidentifiedImageError" in tb
            and "/data/selected/" in tb.replace("\\", "/")
        )
        if snapshot_failure and attempt < MAX_SNAPSHOT_RETRIES:
            repaired = remove_unreadable_selected_images(RUN)
            print(f"Attempt {attempt} hit only a derived snapshot decode failure.")
            print(f"Removed {len(repaired)} unreadable derived snapshot(s); retrying the unchanged frozen runner.")
            continue

        diagnostic = WORKDIR / "Calibration_Count_DIAGNOSTICS.zip"
        with zipfile.ZipFile(diagnostic, "w", zipfile.ZIP_DEFLATED) as z:
            for base in (WORKDIR, RUN):
                if base.exists():
                    for p in base.iterdir():
                        if p.is_file() and p.suffix in (".log", ".json"):
                            z.write(p, str(p.relative_to(WORKDIR)))
            for p in SOURCE.glob("*.py"):
                z.write(p, "runner/" + p.name)
        print("Run did not finish. Return diagnostics:", diagnostic)
        print("Keep WORKDIR to resume verified feature checkpoints; do not change the protocol.")
        if snapshot_failure:
            print("Snapshot retry limit reached:", MAX_SNAPSHOT_RETRIES)
        else:
            print("This was not a derived snapshot decode failure and was intentionally not retried.")
        raise

RESULT = WORKDIR / "Calibration_Count_NEW_CONDITIONS_RESULTS.zip"
status = json.loads((RUN / "STATUS.json").read_text())
if status.get("status") != "COMPLETE" or not RESULT.is_file():
    raise RuntimeError("The complete-result receipt is missing; partial output is not a completed experiment.")
# Add environment setup and runner-source records to the returned archive.
with zipfile.ZipFile(RESULT, "a", zipfile.ZIP_DEFLATED) as z:
    for p in WORKDIR.iterdir():
        if p.is_file() and p.suffix in (".log", ".json"):
            z.write(p, "notebook_runtime/" + p.name)
    for p in SOURCE.iterdir():
        if p.is_file() and p.suffix in (".py", ".txt"):
            z.write(p, "runner/" + p.name)
print("COMPLETE. Return this result file:", RESULT)
print("The analysis retained all scheduled conditions, including unfavorable or unresolved outcomes.")


## 4. Return the result archive

In [ ]:
from IPython.display import display, FileLink
print("Result archive:", RESULT)
if "google.colab" in sys.modules:
    from google.colab import files as colab_files
    colab_files.download(str(RESULT))
else:
    display(FileLink(os.path.relpath(RESULT, Path.cwd())))
